### PART A: Divide our documents into chunks


Import the packages

In [15]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr


In [16]:
%pip install -U plotly

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Configure API key

In [17]:
MODEL = "openai/gpt-oss-120b"
db_name = "vector_db"
load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')
if groq_api_key:
    print(f"GROQ API Key exists and begins {groq_api_key[:8]}")
else:
    print("GROQ API Key not set")

GROQ API Key exists and begins gsk_RVLO


Get data from knowledge base

In [18]:
knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base.")

entire_content = ""

for file in files:
    with open(file, 'r', encoding='utf-8') as f:
        entire_content += f.read()
        entire_content += "\n\n"

print(f"Loaded all files from the knowledge base. {len(entire_content)} characters loaded.")

Found 18 files in the knowledge base.
Loaded all files from the knowledge base. 74896 characters loaded.


In [6]:
encoding = tiktoken.get_encoding("cl100k_base")
tokens = encoding.encode(entire_content)
print(f"Total tokens in the knowledge base: {len(tokens)}")

Total tokens in the knowledge base: 16515


In [7]:
folders = glob.glob("knowledge-base/*", recursive=False)

document  = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader)
    docs = loader.load()
    for doc in docs:
        doc.metadata["doc_type"] = doc_type
    document.extend(docs)
    
    
print(f"Loaded {len(document)} documents from the knowledge base.")

Loaded 18 documents from the knowledge base.


In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(document)
print(f"Split documents into {len(chunks)} chunks.")
print(f"First chunk content: {chunks[0].page_content[:500]}...")  # Print first 500 characters of the first chunk

Split documents into 105 chunks.
First chunk content: ---
title: Distinguished Academic and Research Alumni - PUCIT / FCIT
category: Alumni
campus: Quaid-i-Azam Campus & Allama Iqbal Campus, Lahore
faculty: Faculty of Computing & Information Technology (FCIT)
tags: [alumni, academia, researchers, professors, phd-scholars, higher-education]
---

# Distinguished Academic and Research Alumni — PUCIT / FCIT

A significant number of PUCIT graduates have pursued postgraduate research (MS and PhD) at leading global universities and returned to serve as pr...


In [9]:
chunks[100]

Document(metadata={'source': 'knowledge-base\\teachers\\software-engineering.md', 'doc_type': 'teachers'}, page_content='### Dr. Muddassira Arshad\n* **Designation**: Assistant Professor & Head of FYDP Office\n* **Department**: Department of Software Engineering\n* **Campus**: Quaid-i-Azam Campus (New Campus), Lahore\n* **Email**: `muddassira@pucit.edu.pk`\n* **Official Profile**: [Dr. Muddassira Arshad Faculty Webpage](http://faculty.pucit.edu.pk/muddassira/)\n* **Academic Roles**:\n  * Head of Final Year Design Project (FYDP) Office across FCIT: standardizes Software Requirements Specifications (SRS), Software Design Specifications (SDS), and evaluation rubrics for BS programs.\n  * Member, Discipline Committee & Financial Aid Committee, FCIT.\n  * Advisor and Recruitment Examiner for BPS-17 posts, Punjab Public Service Commission (PPSC).\n* **Research Specialization**: Human-Computer Interaction (HCI), Usability Engineering, Natural Language Processing, Software Quality Assurance, F

### PART B: Make vectors and store in Chroma

You set up a Hugging Face account and got an HF_TOKEN

At this point, you might want to add it to your `.env` file and run `load_dotenv(override=True)`

(This actually shouldn't be required).


In [10]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(chunks, embedding= embeddings, persist_directory=db_name)
print(f"Created vector store with {vectorstore._collection.count()} vectors.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Created vector store with 105 vectors.


In [13]:
collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
print(f"dimensions: {len(sample_embedding)}")

dimensions: 384


### Part C: Visualize!

In [14]:
import re
from pathlib import Path
import pandas as pd


def get_doc_title(text, meta):
  # 1. Check if metadata already has a title
  if meta and meta.get('title'):
    return meta['title']

  # 2. Extract YAML frontmatter 'title: ...' from the text
  yaml_match = re.search(r'^title:\s*(.+)$', text, re.MULTILINE)
  if yaml_match:
    return yaml_match.group(1).strip(' "\'')

  # 3. Extract the first markdown '# Heading' from the text
  h1_match = re.search(r'^#\s+(.+)$', text, re.MULTILINE)
  if h1_match:
    return h1_match.group(1).strip()

  # 4. Fallback to clean filename (e.g. 'careers.md' -> 'Careers')
  if meta and meta.get('source'):
    return Path(meta['source']).stem.replace('-', ' ').title()

  return meta.get('category', 'Document').title()


# Rebuild your records with the title extractor
records = []
for doc, meta in zip(documents, metadatas):
  records.append({
      'title': get_doc_title(doc, meta),
      'category': (
          meta.get('category') or meta.get('doc_type') or 'General'
      ).title(),
      'source': meta.get('source', ''),
      # Clean preview without markdown hashes or yaml for tooltip
      'snippet': doc.strip().split('\n')[-1][:120] + '...',
  })

df_kb = pd.DataFrame(records)

NameError: name 'documents' is not defined

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.manifold import TSNE

# 1. Reduce dimensions of numeric embedding vectors (not records)
# Note: perplexity must be smaller than the number of samples (e.g. 5 or min(30, n_samples - 1))
perplexity_val = min(30, max(1, len(vectors) - 1))
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity_val)
reduced_vectors = tsne.fit_transform(vectors)

# 2. Add coordinates to your DataFrame
df_kb['x'] = reduced_vectors[:, 0]
df_kb['y'] = reduced_vectors[:, 1]

# 3. Create interactive scatter plot with Plotly Express
# (Handles categorical color mapping, legends, and hover cards automatically)
fig = px.scatter(
    df_kb,
    x='x',
    y='y',
    color='category',
    hover_name='title',
    hover_data={'x': False, 'y': False, 'category': True, 'snippet': True},
    title='2D t-SNE Embedding Visualization — PUCIT Knowledge Base',
    color_discrete_map={
        'Teachers': '#2563EB',  # Blue
        'Employees': '#10B981',  # Green
        'Students': '#F59E0B',  # Amber
        'Alumni': '#8B5CF6',  # Purple
    },
    opacity=0.85,
    width=900,
    height=600,
)

fig.update_traces(marker=dict(size=9))
fig.update_layout(
    xaxis_title='t-SNE Dimension 1',
    yaxis_title='t-SNE Dimension 2',
    legend_title_text='Category',
    margin=dict(r=20, b=30, l=30, t=50),
)

fig.show()

In [ ]:
import plotly.express as px
from sklearn.manifold import TSNE

# 1. 3D t-SNE projection
perplexity_val = min(30, max(1, len(vectors) - 1))
tsne = TSNE(n_components=3, random_state=42, perplexity=perplexity_val)
reduced_vectors = tsne.fit_transform(vectors)

# 2. Add 3D coordinates to your DataFrame
df_kb['x'] = reduced_vectors[:, 0]
df_kb['y'] = reduced_vectors[:, 1]
df_kb['z'] = reduced_vectors[:, 2]

# 3. Render interactive 3D plot
fig = px.scatter_3d(
    df_kb,
    x='x',
    y='y',
    z='z',
    color='category',
    hover_name='title',
    hover_data={'x': False, 'y': False, 'z': False, 'snippet': True},
    title='3D Chroma Vector Store Visualization — PUCIT Knowledge Base',
    color_discrete_map={
        'Teachers': '#2563EB',  # Blue
        'Employees': '#10B981',  # Green
        'Students': '#F59E0B',  # Amber
        'Alumni': '#8B5CF6',  # Purple
    },
    opacity=0.85,
    width=950,
    height=750,
)

fig.update_traces(marker=dict(size=5))
fig.update_layout(
    scene=dict(
        xaxis_title='t-SNE 1', yaxis_title='t-SNE 2', zaxis_title='t-SNE 3'
    ),
    legend_title_text='Category',
    margin=dict(r=10, b=10, l=10, t=40),
)

fig.show()

NameError: name 'vectors' is not defined